In [1]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
cd drive/MyDrive/Smart-Warehouse-Delay

/content/drive/MyDrive/Smart-Warehouse-Delay


In [4]:
train = pd.read_csv('data/traffic_V6.csv')
test = pd.read_csv('data/test_traffic_V6.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (249944, 108)
테스트 데이터 크기: (50000, 107)


In [5]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 104


In [6]:
# categorical 처리 (LGBM / XGB용)
train["layout_type"] = train["layout_type"].astype("category").cat.codes
test["layout_type"] = test["layout_type"].astype("category").cat.codes

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [8]:
oof_cat = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

test_cat = np.zeros(len(test))
test_lgb = np.zeros(len(test))
test_xgb = np.zeros(len(test))

In [9]:
for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"\n── Fold {fold + 1} ──")

    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    # =======================
    # 1. CatBoost
    # =======================
    model_cat = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=3,
        loss_function='MAE',
        eval_metric='MAE',
        random_seed=42,
        verbose=0
    )

    model_cat.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        use_best_model=True
    )

    pred_val_cat = model_cat.predict(X_val)
    oof_cat[val_idx] = pred_val_cat
    test_cat += model_cat.predict(test[feature_cols]) / 5

    print(f"Cat MAE: {mean_absolute_error(y_val, pred_val_cat):.4f}")

    # =======================
    # 2. LightGBM
    # =======================
    model_lgb = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model_lgb.fit(X_tr, y_tr)

    pred_val_lgb = model_lgb.predict(X_val)
    oof_lgb[val_idx] = pred_val_lgb
    test_lgb += model_lgb.predict(test[feature_cols]) / 5

    print(f"LGB MAE: {mean_absolute_error(y_val, pred_val_lgb):.4f}")

    # =======================
    # 3. XGBoost
    # =======================
    model_xgb = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=6,
        tree_method='hist',
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model_xgb.fit(X_tr, y_tr)

    pred_val_xgb = model_xgb.predict(X_val)
    oof_xgb[val_idx] = pred_val_xgb
    test_xgb += model_xgb.predict(test[feature_cols]) / 5

    print(f"XGB MAE: {mean_absolute_error(y_val, pred_val_xgb):.4f}")


── Fold 1 ──
Cat MAE: 8.3205
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.390005 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21938
[LightGBM] [Info] Number of data points in the train set: 199955, number of used features: 104
[LightGBM] [Info] Start training from score 18.956003
LGB MAE: 8.1133
XGB MAE: 8.4542

── Fold 2 ──
Cat MAE: 8.3202
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.248701 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21935
[LightGBM] [Info] Number of data points in the train set: 199955, number of used features: 104
[LightGBM] [Info] Start training from score 18.966590
LGB MAE: 8.1775
XGB MAE: 8.4884

── Fold 3 ──
Cat MAE: 8.1834
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.260331 seconds.
You can set `force_col_wise=true` to remove the ov

In [10]:
# 앙상블 (가중 평균)
final_oof = (
    oof_cat * 0.6 +
    oof_lgb * 0.25 +
    oof_xgb * 0.15
)

final_test = (
    test_cat * 0.6 +
    test_lgb * 0.25 +
    test_xgb * 0.15
)

In [11]:
oof_mae = mean_absolute_error(train[TARGET], final_oof)
print(f"Final OOF MAE: {oof_mae:.4f}")

Final OOF MAE: 8.0781


In [12]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: final_test})
submission.to_csv('./submission_V28.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
